In [9]:
import numpy as np
import pandas as pd
import torch

from sklearn.metrics import confusion_matrix

In [10]:
def calculate_classification_metrics(
    targets: torch.Tensor | np.ndarray | list[int],
    predictions: torch.Tensor | np.ndarray | list[int]
) -> dict[str, float]:

    if isinstance(targets, torch.Tensor):
        targets = targets.cpu().numpy()
    
    if isinstance(predictions, torch.Tensor):
        predictions = predictions.cpu().numpy()

    targets = np.array(targets, dtype=int)
    predictions = np.array(predictions, dtype=int)

    cm = confusion_matrix(targets, predictions)
    tn, fp, fn, tp = cm.ravel()

    return {
        'tp': tp,
        'tn': tn,
        'fp': fp,
        'fn': fn,
    }

In [11]:
def sample_results_with_percentage(
    targets: torch.Tensor | np.ndarray | list[int],
    predictions: torch.Tensor | np.ndarray | list[int],
    sample_times = 100,
    results_set_size: int = 500,
    percentage: float = 0.1
) -> dict[str, float]:

    positive_indices = np.where(targets == 1)[0]
    negative_indices = np.where(targets == 0)[0]

    num_positive_samples = int(results_set_size * percentage)
    num_negative_samples = results_set_size - num_positive_samples

    total_tp = 0
    total_tn = 0
    total_fp = 0
    total_fn = 0

    print(f'Number of negative samples: {(targets == 0).sum()}')
    print(f'Number of positive samples: {(targets == 1).sum()}')

    for _ in range(sample_times):
        sampled_positive_indices = np.random.choice(positive_indices, num_positive_samples, replace=False)
        sampled_negative_indices = np.random.choice(negative_indices, num_negative_samples, replace=False)

        sampled_indices = np.concatenate((sampled_positive_indices, sampled_negative_indices))
        np.random.shuffle(sampled_indices)

        sampled_targets = targets[sampled_indices]
        sampled_predictions = predictions[sampled_indices]

        metrics = calculate_classification_metrics(sampled_targets, sampled_predictions)
        
        total_tp += metrics['tp']
        total_tn += metrics['tn']
        total_fp += metrics['fp']
        total_fn += metrics['fn']
    
    average_tp = total_tp / sample_times
    average_tn = total_tn / sample_times
    average_fp = total_fp / sample_times
    average_fn = total_fn / sample_times

    return {
        'tp': average_tp,
        'tn': average_tn,
        'fp': average_fp,
        'fn': average_fn,
        'accuracy': (average_tp + average_tn) / results_set_size,
        'precision': average_tp / (average_tp + average_fp) if (average_tp + average_fp) > 0 else 0,
        'recall': average_tp / (average_tp + average_fn) if (average_tp + average_fn) > 0 else 0,
        'f1_score': 2 * (average_tp / (average_tp + average_fp)) * (average_tp / (average_tp + average_fn)) / ((average_tp / (average_tp + average_fp)) + (average_tp / (average_tp + average_fn))) if (average_tp + average_fp) > 0 and (average_tp + average_fn) > 0 else 0
    }

In [ ]:
RESULTS_PATHS = [
    'results/results.csv'
]

RESULTS_SET_SIZE = 500
SAMPLE_TIMES = 100

PERCENTAGES = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

results_names = []
percentages = []
tps = []
tns = []
fps = []
fns = []
accuracies = []
precisions = []
recalls = []
f1_scores = []

for results_path in RESULTS_PATHS:
    results = pd.read_csv(results_path)
    
    targets = results['label']
    predictions = results['prediction']

    for percentage in PERCENTAGES:
        metrics = sample_results_with_percentage(
            targets=targets,
            predictions=predictions,
            sample_times=SAMPLE_TIMES,
            results_set_size=RESULTS_SET_SIZE,
            percentage=percentage
        )

        print(f"Results for {results_path} with {percentage*100}% positive samples:")
        print(metrics)

        results_names.append(results_path)
        percentages.append(percentage)
        tps.append(metrics['tp'])
        tns.append(metrics['tn'])
        fps.append(metrics['fp'])
        fns.append(metrics['fn'])
        accuracies.append(metrics['accuracy'])
        precisions.append(metrics['precision'])
        recalls.append(metrics['recall'])
        f1_scores.append(metrics['f1_score'])

final_results = pd.DataFrame({
    'results_path': results_names,
    'percentage': percentages,
    'tp': tps,
    'tn': tns,
    'fp': fps,
    'fn': fns,
    'accuracy': accuracies,
    'precision': precisions,
    'recall': recalls,
    'f1_score': f1_scores
})

final_results.to_csv('results/sampled_results.csv', index=False)
print(final_results)

Number of negative samples: 675
Number of positive samples: 6180
Results for results/results.csv with 10.0% positive samples:
{'tp': np.float64(46.91), 'tn': np.float64(208.86), 'fp': np.float64(241.14), 'fn': np.float64(3.09), 'accuracy': np.float64(0.51154), 'precision': np.float64(0.16285367123763236), 'recall': np.float64(0.9381999999999999), 'f1_score': np.float64(0.27753290933293895)}
Number of negative samples: 675
Number of positive samples: 6180
Results for results/results.csv with 20.0% positive samples:
{'tp': np.float64(93.87), 'tn': np.float64(185.99), 'fp': np.float64(214.01), 'fn': np.float64(6.13), 'accuracy': np.float64(0.55972), 'precision': np.float64(0.3048915161751332), 'recall': np.float64(0.9387000000000001), 'f1_score': np.float64(0.46028243601059143)}
Number of negative samples: 675
Number of positive samples: 6180
Results for results/results.csv with 30.0% positive samples:
{'tp': np.float64(141.3), 'tn': np.float64(161.73), 'fp': np.float64(188.27), 'fn': np.